In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import ast

In [2]:
dataset=pd.read_csv("movies_metadata.csv")

In [3]:
dataset.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='str')

In [4]:
dataset.shape

(45466, 24)

In [5]:
dataset.isnull().sum()

adult                        0
belongs_to_collection    40972
budget                       0
genres                       0
homepage                 37684
id                           0
imdb_id                     17
original_language           11
original_title               0
overview                   954
popularity                   5
poster_path                386
production_companies         3
production_countries         3
release_date                87
revenue                      6
runtime                    263
spoken_languages             6
status                      87
tagline                  25054
title                        6
video                        6
vote_average                 6
vote_count                   6
dtype: int64

In [6]:
dataset.duplicated().sum()

np.int64(13)

In [7]:
dataset= dataset.drop_duplicates().reset_index(drop=True)

In [8]:
dataset= dataset[['title','overview','genres','tagline','vote_average','popularity']]
dataset

,title,overview,genres,tagline,vote_average,popularity
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",NaN,7.7,21.946943
1,Jumanji,When siblings Judy and Peter discover an encha...,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",Roll the dice and unleash the excitement!,6.9,17.015539
2,Grumpier Old Men,A family wedding reignites the ancient feud be...,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",Still Yelling. Still Fighting. Still Ready for...,6.5,11.7129
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...","[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",Friends are the people who let you be yourself...,6.1,3.859495
4,Father of the Bride Part II,Just when George Banks has recovered from his ...,"[{'id': 35, 'name': 'Comedy'}]",Just When His World Is Back To Normal... He's ...,5.7,8.387519
...,...,...,...,...,...,...
45448,Subdue,Rising and falling between a man and woman.,"[{'id': 18, 'name': 'Drama'}, {'id': 10751, 'n...",Rising and falling between a man and woman,4.0,0.072051
45449,Century of Birthing,An artist struggles to finish his work while a...,"[{'id': 18, 'name': 'Drama'}]",NaN,9.0,0.178241
45450,Betrayal,"When one of her hits goes wrong, a professiona...","[{'id': 28, 'name': 'Action'}, {'id': 18, 'nam...",A deadly game of wits.,3.8,0.903007
45451,Satan Triumphant,"In a small town live two brothers, one a minis...",[],NaN,0.0,0.003503


In [9]:
dataset.dropna(subset=['title'],inplace=True)

In [10]:
dataset['overview']=dataset['overview'].fillna('')

In [11]:
dataset['tagline']=dataset['tagline'].fillna('')

In [12]:
dataset.iloc[0]['genres']

"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]"

In [13]:
dataset['genres']=dataset['genres'].apply(lambda x :" ".join([i['name'] for i in ast.literal_eval(x)]))

In [14]:
# combining columns overview, genres, tagline into a 1 single column

dataset['tags']=dataset['overview']+" "+dataset['genres']+" "+dataset['tagline']
dataset

,title,overview,genres,tagline,vote_average,popularity,tags
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...",Animation Comedy Family,,7.7,21.946943,"Led by Woody, Andy's toys live happily in his ..."
1,Jumanji,When siblings Judy and Peter discover an encha...,Adventure Fantasy Family,Roll the dice and unleash the excitement!,6.9,17.015539,When siblings Judy and Peter discover an encha...
2,Grumpier Old Men,A family wedding reignites the ancient feud be...,Romance Comedy,Still Yelling. Still Fighting. Still Ready for...,6.5,11.7129,A family wedding reignites the ancient feud be...
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",Comedy Drama Romance,Friends are the people who let you be yourself...,6.1,3.859495,"Cheated on, mistreated and stepped on, the wom..."
4,Father of the Bride Part II,Just when George Banks has recovered from his ...,Comedy,Just When His World Is Back To Normal... He's ...,5.7,8.387519,Just when George Banks has recovered from his ...
...,...,...,...,...,...,...,...
45448,Subdue,Rising and falling between a man and woman.,Drama Family,Rising and falling between a man and woman,4.0,0.072051,Rising and falling between a man and woman. Dr...
45449,Century of Birthing,An artist struggles to finish his work while a...,Drama,,9.0,0.178241,An artist struggles to finish his work while a...
45450,Betrayal,"When one of her hits goes wrong, a professiona...",Action Drama Thriller,A deadly game of wits.,3.8,0.903007,"When one of her hits goes wrong, a professiona..."
45451,Satan Triumphant,"In a small town live two brothers, one a minis...",,,0.0,0.003503,"In a small town live two brothers, one a minis..."


In [15]:
# converting tags column into mathematical values using NLP

# starting with data preprocessing like : punctuation removal -> stop word removal -> lemmatization -> tokenization

In [16]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

In [17]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kushi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\kushi\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [18]:
stop_words= set(stopwords.words('english'))
lemmatizer= WordNetLemmatizer()

In [19]:
def preprocess_text(text):

    # 1. Lowercase
    text= str(text).lower()

    # 2. Remove punctuation
    text= re.sub(r'[^a-zA-Z\s]', '',text)

    # 3. Tokenization
    words=text.split()

    # 4. Lemmatization & Stop-word removal
    words=[lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    return " ".join(words)


In [20]:
dataset['tags']=dataset['tags'].apply(preprocess_text)

In [21]:
dataset=dataset.reset_index(drop=True)

In [22]:
indices= pd.Series(dataset.index, index= dataset['title']).drop_duplicates()
indices

title
Toy Story                          0
Jumanji                            1
Grumpier Old Men                   2
Waiting to Exhale                  3
Father of the Bride Part II        4
                               ...  
Subdue                         45442
Century of Birthing            45443
Betrayal                       45444
Satan Triumphant               45445
Queerama                       45446
Length: 45447, dtype: int64

In [23]:
# vectorization of tags column using tf-idf

from sklearn.feature_extraction.text import TfidfVectorizer

In [24]:
tfidf= TfidfVectorizer(max_features=50000, ngram_range=(1,2), stop_words='english')

In [25]:
tfidf_matrix= tfidf.fit_transform(dataset['tags'])

In [26]:
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1548114 stored elements and shape (45447, 50000)>

In [27]:
# Creating cosine similarity
from sklearn.metrics.pairwise import linear_kernel

In [28]:
def recommend(title, top_n=10):

    if title not in indices:
        return f"Movie '{title}' not found in dataset."

    idx=indices[title]

    sim_score=linear_kernel(tfidf_matrix[idx], tfidf_matrix).flatten()
    sim_indices=sim_score.argsort()[::-1][1:top_n+1]
    return dataset['title'].iloc[sim_indices]

In [31]:
recommend('Avatar')

26547                  Avatar 2
13883      The Inhabited Island
26553            Thor: Ragnarok
43431    Moontrap: Target Earth
14119      The Three Musketeers
9949         A Trip to the Moon
43983       Nightmare City 2035
28403    France société anonyme
20372                    Désiré
28633      Stand by Me Doraemon
Name: title, dtype: str

In [32]:
import pickle

pickle.dump(tfidf_matrix,open('tfidf_matrix.pkl','wb'))

pickle.dump(indices,open('indices.pkl','wb'))

dataset.to_pickle('df.pkl')

pickle.dump(tfidf,open('tfidf.pkl','wb'))